[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

In [3]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [4]:
import torch
import torch.nn as nn

In [26]:
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        # pass  # router + experts
        # d_ff: dim of hidden layer
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(d_model, d_ff),  # expand:   d_model → d_ff
                    nn.ReLU(),
                    nn.Linear(d_ff, d_model)    # contract: d_ff → d_model
                )
                for _ in range(num_experts)
            ]
        )
        self.top_k = top_k

    def forward(self, x):
        # pass  # route tokens to top-k experts
        router_score = self.router(x)
        # (B, S, D) -> (B, S, num_experts)
        # use torch.topk function to pick
        topk_vals, topk_idx = torch.topk(router_score, self.top_k, dim = -1)
        # topk_vals: (B, S, k)  — the k highest scores per token
        # topk_idx:  (B, S, k)  — which experts those are
        topk_weights = torch.softmax(topk_vals, dim = -1)
        output = torch.zeros_like(x)

        for i in range(self.top_k):
          topi = topk_idx[:, :, i] # e.g. all top0 (B, S)
          weights = topk_weights[:, :, i] # corresponding weights -> (B, S)
          for e in range(len(self.experts)):
            # find which token this e is the top0
            mask = (topi == e).bool() #(B, S)
            # take these tokens:
            if mask.any():
              x_selected = x[mask] # num_selected, D


        # x:    (B, S, D)          e.g. (2, 4, 8)
        # mask: (B, S)             bool, e.g. True for tokens routed to expert e

        # x[mask]: (num_true, D)   e.g. (5, 8)  if 5 tokens were True


              exp_out = self.experts[e](x_selected)  # num_selected, D
              flat_out = weights[mask].unsqueeze(-1) * exp_out # (num_selected, 1) & (num_selected, D)
              # selected topi weights * corresponding experts' outputs, (num_selected, D)
              # this weight will boardcast across all D dimensions
              output[mask] += flat_out



        return output






In [27]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

Output: torch.Size([2, 8, 32])
Params: 16900


In [28]:
# ✅ SUBMIT
from torch_judge import check
check('moe')


🧪 Testing: Mixture of Experts (MoE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (10.9ms)
  ✅ [2/4] Has router and experts (2.6ms)
  ✅ [3/4] Router logits shape (2.9ms)
  ✅ [4/4] Gradient flow (40.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (56.9ms total)
  Progress saved. Run status() to see your dashboard.

